# 18-stop cinematic flyover: Earth Studio -> MP4
How the camera moves:
- **Intro:** descend from orbit into the first city.
- **At each stop:** an oblique orbit (tilted camera, slow pan around the city) that zooms in.
- **Short hops** (< `HORIZ_MAX_KM`): low, tilted, forward-looking flight to the next city.
- **Long hops:** pull up, rise toward space (camera tilts down to look straight at the ground), then dive back into an oblique view.
- **Outro:** pull back out to the globe.

Steps:
1. Run cells 1–4. This downloads `flyover.esp` and a `flyover.kml` tour.
2. In Earth Studio, open `flyover.esp` and Render.
3. Run cell 5 and upload the render (`.zip` of JPEGs or `.mp4`). You get `flyover_final.mp4`.

The camera keeps aiming at each city through Earth Studio's **Camera Target**, so the orbit always stays centered.
If the from-scratch ESP won't import, build a template:
1. In Earth Studio, make a small project with a couple of camera keyframes and a Camera Target track.
2. Save it as `template.esp`.
3. Set `USE_TEMPLATE=True` in cell 3.

The notebook fills the keyframes into that file's own schema.

In [ ]:
# 1 | config
import json, math, copy, os, glob, zipfile, subprocess, shutil

# name, lat, lon, ground elevation (m, approx; keeps the camera above terrain in Earth Studio)
STOPS = [
    ("Coeur d'Alene, ID",     47.6777, -116.7805,  670),
    ("Deer Park, WA",         47.9543, -117.4769,  645),
    ("Winlock, WA",           46.4910, -122.9379,  100),
    ("Ellensburg, WA",        46.9965, -120.5478,  470),
    ("Portland, OR",          45.5152, -122.6784,   50),
    ("Cathlamet, WA",         46.2032, -123.3832,   20),
    ("McKenzie Bridge, OR",   44.1754, -122.1606,  420),
    ("Lyle, WA",              45.6962, -121.2862,   60),
    ("Springfield, IL",       39.7817,  -89.6501,  180),
    ("Santa Fe, NM",          35.6870, -105.9378, 2130),
    ("Fort Collins, CO",      40.5853, -105.0844, 1525),
    ("Huntsville, AL",        34.7304,  -86.5861,  190),
    ("Bangkok, Thailand",     13.7563,  100.5018,    5),
    ("Naypyidaw, Myanmar",    19.7633,   96.0785,  115),
    ("Kathmandu, Nepal",      27.7172,   85.3240, 1400),
    ("Dhaka, Bangladesh",     23.8103,   90.4125,   10),
    ("Thimphu, Bhutan",       27.4728,   89.6390, 2330),
    ("Enschede, Netherlands", 52.2215,    6.8937,   40),
]

# --- timing ---
TOTAL_SEC     = 120     # whole video (raise to 150-180 for an even slower feel)
FPS           = 30
INTRO_SEC     = 4.0     # descend from space into stop 1
OUTRO_SEC     = 3.0     # pull out to globe at the end
ORBIT_SEC     = 3.6     # time circling each city
MIN_LEG_SEC   = 1.8     # floor for short hops

# --- camera at each stop (LookAt terms: range = distance to target) ---
ORBIT_DEG     = 90      # degrees of pan around each city
ORBIT_TILT    = 62      # 0 = straight down, 90 = horizon
RANGE_IN      = 9_000   # m, range when orbit starts
RANGE_OUT     = 4_200   # m, range when orbit ends (zoom in)

# --- transit ---
HORIZ_MAX_KM  = 600     # legs shorter than this are low "horizontal" flyovers
CRUISE_TILT   = 68      # tilt during low flyovers (looking forward)
HORIZ_APEX_FACTOR = 0.20    # low flyover cruise range ~ 0.20 x leg distance
HIGH_APEX_FACTOR  = 0.9     # long hops rise to ~0.9 x leg distance
MAX_APEX_M    = 9_000_000
SAMPLE_HZ     = 4       # camera keyframes per second (dense = smooth curves)

WIDTH, HEIGHT = 1920, 1080
REORDER_ASIA  = False   # True: Bangkok>Naypyidaw>Dhaka>Thimphu>Kathmandu

# --- video overlay (cell 5) ---
LABELS        = True
ATTRIBUTION   = "Google Earth"   # required by Earth Studio terms; "" if already burned in

if REORDER_ASIA:
    order = ["Bangkok", "Naypyidaw", "Dhaka", "Thimphu", "Kathmandu"]
    asia = [next(s for s in STOPS if s[0].startswith(o)) for o in order]
    STOPS = [s for s in STOPS if s not in asia]
    i = [s[0] for s in STOPS].index("Enschede, Netherlands")
    STOPS = STOPS[:i] + asia + STOPS[i:]

In [ ]:
# 2 | camera path: LookAt(t) -> (target lat, lon, elev, range, tilt, heading)
R_E = 6_371_000.0
rad, deg = math.radians, math.degrees
smooth  = lambda s: s*s*(3 - 2*s)
smooth5 = lambda s: s*s*s*(s*(6*s - 15) + 10)
lerp    = lambda a, b, s: a + (b - a)*s

def haversine_m(a, b):
    p1, p2 = rad(a[0]), rad(b[0]); dp, dl = p2 - p1, rad(b[1] - a[1])
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R_E*math.asin(math.sqrt(h))

def bearing(a, b):
    p1, p2, dl = rad(a[0]), rad(b[0]), rad(b[1] - a[1])
    y = math.sin(dl)*math.cos(p2); x = math.cos(p1)*math.sin(p2) - math.sin(p1)*math.cos(p2)*math.cos(dl)
    return deg(math.atan2(y, x)) % 360

def dest(lat, lon, brg, dist):
    p1, l1, b, d = rad(lat), rad(lon), rad(brg), dist/R_E
    p2 = math.asin(math.sin(p1)*math.cos(d) + math.cos(p1)*math.sin(d)*math.cos(b))
    l2 = l1 + math.atan2(math.sin(b)*math.sin(d)*math.cos(p1), math.cos(d) - math.sin(p1)*math.sin(p2))
    return deg(p2), (deg(l2) + 540) % 360 - 180

def ang_lerp(a, b, s):            # shortest-way heading blend, unwrapped
    d = (b - a + 540) % 360 - 180
    return a + d*s

n = len(STOPS)
P = [(s[1], s[2]) for s in STOPS]
dists = [haversine_m(P[i], P[i+1]) for i in range(n-1)]
fixed = INTRO_SEC + OUTRO_SEC + ORBIT_SEC*n + MIN_LEG_SEC*(n-1)
assert fixed < TOTAL_SEC, f"TOTAL_SEC too short (need > {fixed:.0f}s)"
w = [math.sqrt(d) for d in dists]
leg_sec = [MIN_LEG_SEC + (TOTAL_SEC - fixed)*x/sum(w) for x in w]

# segments: (kind, t0, t1, data)
segs, t, h = [], 0.0, None
h_arrive = bearing(P[0], P[1])
segs.append(("intro", t, t + INTRO_SEC, 0)); t += INTRO_SEC
orbit_windows = []
for i in range(n):
    h0 = h_arrive
    segs.append(("orbit", t, t + ORBIT_SEC, (i, h0))); orbit_windows.append((STOPS[i][0], t, t + ORBIT_SEC))
    t += ORBIT_SEC
    h_end = h0 + ORBIT_DEG
    if i < n - 1:
        segs.append(("leg", t, t + leg_sec[i], (i, h_end))); t += leg_sec[i]
        h_arrive = bearing(P[i], P[i+1])
    else:
        segs.append(("outro", t, t + OUTRO_SEC, (i, h_end))); t += OUTRO_SEC
TOTAL = t

def lookat(tq):
    for kind, t0, t1, data in segs:
        if t0 <= tq <= t1 + 1e-9: break
    s = min(1.0, max(0.0, (tq - t0)/(t1 - t0)))
    if kind == "intro":
        lat, lon = P[0]; e = smooth5(s)
        r = math.exp(lerp(math.log(6_000_000), math.log(RANGE_IN), e))
        return lat, lon, STOPS[0][3], r, ORBIT_TILT*smooth(s), bearing(P[0], P[1])
    if kind == "orbit":
        i, h0 = data; lat, lon = P[i]
        r = math.exp(lerp(math.log(RANGE_IN), math.log(RANGE_OUT), smooth(s)))
        return lat, lon, STOPS[i][3], r, ORBIT_TILT, h0 + ORBIT_DEG*s
    if kind == "outro":
        i, h_end = data; lat, lon = P[i]; e = smooth5(s)
        r = math.exp(lerp(math.log(RANGE_OUT), math.log(8_000_000), e))
        return lat, lon, STOPS[i][3], r, ORBIT_TILT*(1 - smooth(s)), h_end + 30*s
    # leg
    i, h_end = data; A, B = P[i], P[i+1]; d = dists[i]
    u = smooth5(s)
    lat = lerp(A[0], B[0], u); lon = lerp(A[1], B[1], u)       # no antimeridian crossing
    elev = lerp(STOPS[i][3], STOPS[i+1][3], u)
    bump = math.sin(math.pi*s)
    base = math.exp(lerp(math.log(RANGE_OUT), math.log(RANGE_IN), s))
    horiz = d/1000 < HORIZ_MAX_KM
    apex = min(MAX_APEX_M, max(RANGE_IN*2, (HORIZ_APEX_FACTOR if horiz else HIGH_APEX_FACTOR)*d))
    r = math.exp(lerp(math.log(base), math.log(apex), bump))
    tilt = (lerp(ORBIT_TILT, CRUISE_TILT, bump) if horiz else ORBIT_TILT*(1 - bump**0.6))
    ahead = (lerp(A[0], B[0], min(1, u + 0.02)), lerp(A[1], B[1], min(1, u + 0.02)))
    travel = bearing((lat, lon), ahead) if haversine_m((lat, lon), ahead) > 1 else bearing(A, B)
    hd = ang_lerp(h_end, travel, smooth(min(1, s/0.35)))          # swing from orbit heading to forward
    return lat, lon, elev, r, tilt, hd

def camera(la):                     # LookAt -> camera position (behind target)
    lat, lon, elev, r, tilt, hd = la
    g = r*math.sin(rad(tilt))
    clat, clon = dest(lat, lon, hd + 180, g)
    return clat, clon, elev + r*math.cos(rad(tilt))

N = int(round(TOTAL*SAMPLE_HZ))
samples = []
for k in range(N + 1):
    tq = TOTAL*k/N
    la = lookat(tq); cam = camera(la)
    samples.append((tq, la, cam))

DURATION_FR = round(TOTAL*FPS)
for (name, *_), d, s in zip(STOPS[1:], dists, leg_sec):
    print(f"-> {name:24s} {d/1000:8.0f} km  {s:5.2f}s  {'low flyover' if d/1000 < HORIZ_MAX_KM else 'high arc'}")
low = min(c[2] - la[2] for _, la, c in samples)
print(f"total {TOTAL:.1f}s | {len(samples)} camera keyframes | {DURATION_FR} frames | min clearance {low:.0f} m")

In [ ]:
# 2b | quick preview of the path (map + altitude)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot([c[1] for *_, c in samples], [c[0] for *_, c in samples], lw=0.8, label="camera")
ax[0].scatter([p[1] for p in P], [p[0] for p in P], c="r", s=12, zorder=3, label="stops")
ax[0].set_title("ground track"); ax[0].legend(); ax[0].set_xlabel("lon"); ax[0].set_ylabel("lat")
ax[1].semilogy([t for t, *_ in samples], [c[2] for *_, c in samples], lw=0.8)
for _, a, b in orbit_windows: ax[1].axvspan(a, b, color="orange", alpha=0.25)
ax[1].set_title("camera altitude (orange = orbits)"); ax[1].set_xlabel("s"); ax[1].set_ylabel("m")
plt.tight_layout(); plt.show()

In [ ]:
# 3 | build flyover.esp + flyover.kml
try:
    from google.colab import files
except ImportError:
    files = None

DEF_RANGE = {"longitude": (-180, 180), "latitude": (-89.9999, 89.9999), "altitude": (1, 65_117_481),
             "longitudePOI": (-180, 180), "latitudePOI": (-89.9999, 89.9999), "altitudePOI": (1, 65_117_481)}

def channel_values(kind):
    out = []
    for tq, la, cam in samples:
        out.append((tq, {"longitude": cam[1], "latitude": cam[0], "altitude": cam[2],
                         "longitudePOI": la[1], "latitudePOI": la[0], "altitudePOI": max(1, la[2])}[kind]))
    return out

def make_kfs(kind, lo, hi, proto=None, frames_mode=False):
    kfs = []
    for tq, v in channel_values(kind):
        k = copy.deepcopy(proto) if proto else {}
        k["time"]  = round(tq*FPS) if frames_mode else tq/TOTAL
        k["value"] = (v - lo)/(hi - lo)
        kfs.append(k)
    return kfs

def build_esp_from_scratch():
    def attr(k):
        lo, hi = DEF_RANGE[k]
        return {"type": k, "value": {"minValueRange": lo, "maxValueRange": hi}, "keyframes": make_kfs(k, lo, hi)}
    return {
        "modelVersion": 16,
        "settings": {"name": "18-stop flyover", "frameRate": FPS,
                     "dimensions": {"width": WIDTH, "height": HEIGHT},
                     "duration": DURATION_FR, "timeFormat": "frames"},
        "scenes": [{
            "animationModel": {"roving": False, "logarithmic": False, "groupedPosition": True},
            "duration": DURATION_FR,
            "attributes": [
                {"type": "cameraGroup", "attributes": [
                    {"type": "cameraPositionGroup", "attributes": [
                        {"type": "position", "attributes": [attr("longitude"), attr("latitude"), attr("altitude")]}]}]},
                {"type": "cameraTargetEffect", "attributes": [
                    {"type": "poi", "attributes": [attr("longitudePOI"), attr("latitudePOI"), attr("altitudePOI")]},
                    {"type": "influence", "value": {"minValueRange": 0, "maxValueRange": 1},
                     "keyframes": [{"time": 0, "value": 1}, {"time": 1, "value": 1}]}]},
            ],
        }],
        "playbackManager": {"range": {"start": 0, "end": DURATION_FR}},
    }

def inject_into_template(esp):
    hits = []
    def walk(o):
        if isinstance(o, dict):
            k = o.get("type")
            if k in DEF_RANGE and "keyframes" in o:
                v = o.get("value", {}) or {}
                lo, hi = v.get("minValueRange", DEF_RANGE[k][0]), v.get("maxValueRange", DEF_RANGE[k][1])
                old = o["keyframes"]
                proto = {a: b for a, b in old[0].items() if a not in ("time", "value")} if old else None
                o["keyframes"] = make_kfs(k, lo, hi, proto, any(x.get("time", 0) > 1.0 for x in old))
                hits.append(k)
            for v in o.values(): walk(v)
        elif isinstance(o, list):
            for v in o: walk(v)
    walk(esp)
    assert {"latitude", "longitude", "altitude"} <= set(hits), f"template missing camera keyframes: {hits}"
    if not {"latitudePOI", "longitudePOI"} <= set(hits):
        print("WARNING: template has no Camera Target track -> camera won't aim at cities. Add one and re-export.")
    esp.setdefault("settings", {}).update(duration=DURATION_FR, frameRate=FPS)
    for sc in esp.get("scenes", []): sc["duration"] = DURATION_FR
    esp.setdefault("playbackManager", {})["range"] = {"start": 0, "end": DURATION_FR}
    print("filled:", sorted(set(hits)))
    return esp

USE_TEMPLATE = False
if USE_TEMPLATE and not os.path.exists("template.esp") and files:
    files.upload()
if USE_TEMPLATE and os.path.exists("template.esp"):
    esp = inject_into_template(json.load(open("template.esp"))); print("ESP from template.esp")
else:
    esp = build_esp_from_scratch(); print("ESP from scratch")
json.dump(esp, open("flyover.esp", "w"))

# KML tour: LookAt natively does orbit/tilt/heading -> Google Earth Pro > Tools > Movie Maker
steps, prev = [], 0.0
for tq, (lat, lon, elev, r, tilt, hd), _ in samples:
    dur = max(tq - prev, 1e-3); prev = tq
    steps.append(f"<gx:FlyTo><gx:duration>{dur:.3f}</gx:duration><gx:flyToMode>smooth</gx:flyToMode>"
                 f"<LookAt><longitude>{lon:.6f}</longitude><latitude>{lat:.6f}</latitude><altitude>0</altitude>"
                 f"<heading>{(hd + 540) % 360 - 180:.2f}</heading><tilt>{tilt:.2f}</tilt><range>{r:.1f}</range>"
                 f"<altitudeMode>relativeToGround</altitudeMode></LookAt></gx:FlyTo>")
pins = "".join(f"<Placemark><name>{s[0].replace('&','&amp;')}</name><Point><coordinates>{s[2]},{s[1]},0</coordinates></Point></Placemark>"
               for s in STOPS)
open("flyover.kml", "w").write(f'''<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2" xmlns:gx="http://www.google.com/kml/ext/2.2">
<Document><name>18-stop flyover</name>{pins}
<gx:Tour><name>Flyover tour</name><gx:Playlist>{''.join(steps)}</gx:Playlist></gx:Tour>
</Document></kml>''')
print("wrote flyover.esp", os.path.getsize("flyover.esp")//1024, "KB | flyover.kml")

In [ ]:
# 4 | download -> open flyover.esp in Earth Studio and Render
files.download("flyover.esp")
files.download("flyover.kml")

### Earth Studio
Open the project and check the preview: the camera should circle each city. Then **Render**.
- **Advanced** render (full quality JPEGs) gives you a `.zip`.
- **Quick** render gives you an `.mp4`.

Then run cell 5.

In [ ]:
# 5 | upload Earth Studio render (.zip of JPEGs or .mp4) -> flyover_final.mp4
FONT = subprocess.run(["fc-match", "-f", "%{file}", "DejaVu Sans:bold"], capture_output=True, text=True).stdout.strip()

def overlay_filter():
    os.makedirs("labels", exist_ok=True); f = []
    if LABELS:
        for i, (name, a, b) in enumerate(orbit_windows):
            p = os.path.abspath(f"labels/{i:02d}.txt"); open(p, "w").write(name)
            fade = f"if(lt(t,{a+0.4:.2f}),(t-{a:.2f})/0.4,if(gt(t,{b-0.4:.2f}),({b:.2f}-t)/0.4,1))"
            f.append(f"drawtext=fontfile={FONT}:textfile={p}:fontsize=h/17:fontcolor=white"
                     f":shadowcolor=black@0.7:shadowx=3:shadowy=3:x=w*0.05:y=h*0.82"
                     f":alpha='{fade}':enable='between(t,{a:.2f},{b:.2f})'")
    if ATTRIBUTION:
        p = os.path.abspath("labels/attrib.txt"); open(p, "w").write(ATTRIBUTION)
        f.append(f"drawtext=fontfile={FONT}:textfile={p}:fontsize=h/45:fontcolor=white@0.85"
                 f":shadowcolor=black@0.6:shadowx=2:shadowy=2:x=w-tw-24:y=h-th-20")
    return ",".join(f) or "null"

up = files.upload()
src = next(iter(up))
enc = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", "-preset", "medium", "-movflags", "+faststart"]

if src.lower().endswith(".zip"):
    shutil.rmtree("frames", ignore_errors=True); shutil.rmtree("seq", ignore_errors=True)
    zipfile.ZipFile(src).extractall("frames")
    jpgs = sorted(glob.glob("frames/**/*.jp*g", recursive=True))
    assert jpgs, "no JPEGs in zip"
    print(f"{len(jpgs)} frames (expected ~{DURATION_FR})")
    os.makedirs("seq")
    for i, p in enumerate(jpgs): os.symlink(os.path.abspath(p), f"seq/{i:05d}.jpg")
    inp = ["-framerate", str(FPS), "-i", "seq/%05d.jpg"]
else:
    inp = ["-i", src]

subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *inp, "-vf", overlay_filter(), "-r", str(FPS), *enc,
                "flyover_final.mp4"], check=True)
print(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=width,height",
                      "-of", "default=nw=1", "flyover_final.mp4"], capture_output=True, text=True).stdout)
files.download("flyover_final.mp4")